# 07 - FastChem-Truth Recovery (Guillot/petitRADTRANS PT profile)

The single notebook whose job is to answer **"can the emulator stand in for
the classical chemistry inside a NUTS retrieval, with no contamination from
the classical-vs-classical residual?"**

Framing
-------
Live FastChem is the training oracle, but it is a non-differentiable subprocess
and cannot run inside the NUTS kernel. So:

* **FastChem** generates the mock spectra (truth).
* **Emulator** runs the NUTS chain (the only differentiable backend).
* **ExoGibbs** is shown only as a forward-VMR overlay - it is *not* in any
  recovery chain. That removes the FastChem-vs-ExoGibbs residual from the
  replacement test entirely, which is what the original question asked for.

PT profile
----------
This version uses the **Piette & Madhusudhan 2019 / Guillot 2010** profile
(``src.models.standalone_inference.guillot_temperature``), the same shape
the bundle was trained on (see ``temperature_profiles.analytic_sampler`` in
``config/fastchem.json`` and ``src/data_generation/sampling.py``).
petitRADTRANS uses the same family.

Parameterization (no upper-atmosphere modification, no smoothing - the
alpha=0 corner of the analytic family the bundle saw):

    T^4(P) = (3/4) T_int^4 (2/3 + delta P)
           + (3/4) T_eq^4 [2/3 + 1/(gamma sqrt(3))
           +  (gamma/sqrt(3) - 1/(gamma sqrt(3))) exp(-gamma sqrt(3) delta P)]

Free for retrieval:
* ``T_int_k``     - internal temperature
* ``T_eq_k``      - equilibrium temperature
* ``log10_gamma`` - log10(kappa_V / kappa_IR)

Pinned at deployment-typical values (and rolled into ``logg``):
* ``log10_delta = 0`` - kappa_IR / g, sets the IR optical-depth scale.
  Free ``logg`` plus pinned delta is the same dimensionality as the OG
  ``T0 + alpha`` parameterization, with profiles that match the training
  family.

Parameter coverage
------------------
The forward-spectrum stack mirrors the upstream tutorial
``og_equilibrium_chemistry_transformer.ipynb`` (Hajime Kawahara, ExoJAX v2.1):
CO line opacity from ExoMol + H2-H2 CIA, ``logZ`` retrieved as a single
metallicity scaling on C and O, ``sigmain`` sampled, NUTS prior
``logZ ~ U(-1, 1)``.

The recovery test runs at four ``logZ`` truths spanning the prior
(``-0.5, 0.0, +0.5, +0.7``) plus one cooler-atmosphere variant, so the
surrogate is gated across the full prior breadth, not just at solar.


## 1. Wavenumber grid and CO opacity

Match the OG tutorials CO band (22920-23000 AA) and `OpaPremodit` setup.

In [ ]:
# ExoJAX runs in float64; emulator is float32 internally and JAX downcasts at
# the boundary. This matches training precision.
from jax import config
config.update("jax_enable_x64", True)

import os
import sys
import time
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from numpy.random import default_rng

# Allow override via VULCAN_PROJECT_ROOT (used when nbconvert runs from /tmp).
_env_root = os.environ.get("VULCAN_PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "exojax_demo":
        PROJECT_ROOT = PROJECT_ROOT.parent
print("PROJECT_ROOT :", PROJECT_ROOT)

MODEL = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), f"src not found at {DIST_ROOT / 'src'}"
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

_STYLE = (PROJECT_ROOT / "exojax_demo" / "science.mplstyle")
if _STYLE.exists():
    plt.style.use(str(_STYLE))

print("jax backend  :", jax.default_backend(), "| devices:", jax.devices())
print("MODEL        :", MODEL)
print("BUNDLE_PATH  :", BUNDLE_PATH)


In [ ]:
from exojax.utils.grids import wavenumber_grid

nu_grid, wav, resolution = wavenumber_grid(
    22920.0, 23000.0, 1500, unit="AA", xsmode="premodit"
)
print(f"R = {resolution:.0f}  N(nu) = {len(nu_grid)}")


In [ ]:
from exojax.database.exomol.api import MdbExomol
from exojax.opacity import OpaPremodit

mdb = MdbExomol(".database/CO/12C-16O/Li2015", nurange=nu_grid)
opa = OpaPremodit(
    mdb=mdb,
    nu_grid=nu_grid,
    auto_trange=[500.0, 2500.0],
    dit_grid_resolution=1.0,
    allow_32bit=True,
)


## 2. Radiative transfer setup

Pressure span 1e-5 to 10 bar (matches OG, sits inside the bundle trained
pressure union). NLAYER=60 is the upper bound of the bundles
`num_levels_range`; the bundle rejects calls outside that range so we pin to
the trained maximum for resolution. Temperature range covers the Guillot
truths through the prior bounds.

In [ ]:
from exojax.rt import ArtEmisPure

NLAYER = 60  # bundle trained range is [40, 60]; use the upper bound for resolution.

art = ArtEmisPure(
    nu_grid=nu_grid,
    pressure_btm=1.0e1,
    pressure_top=1.0e-5,
    nlayer=NLAYER,
    rtsolver="ibased",
    nstream=8,
)
art.change_temperature_range(300.0, 2500.0)

print("RT grid     :", f"nlayer={NLAYER}  P=[{art.pressure.min():.3e}, {art.pressure.max():.3e}] bar")


In [ ]:
from exojax.database.contdb import CdbCIA
from exojax.opacity import OpaCIA
from exojax.postproc.specop import SopRotation, SopInstProfile
from exojax.utils.instfunc import resolution_to_gaussian_std

cdb = CdbCIA(".database/H2-H2_2011.cia", nurange=nu_grid)
opacia = OpaCIA(cdb, nu_grid=nu_grid)
sop_rot = SopRotation(nu_grid, vsini_max=100.0)
sop_inst = SopInstProfile(nu_grid, vrmax=1000.0)

RES_INST = 70000.0
BETA_INST = resolution_to_gaussian_std(RES_INST)
U1, U2 = 0.0, 0.0

# Observation grid: every 5th point of nu_grid, drop the last 50 to avoid edge artifacts.
nu_obs = np.asarray(nu_grid[::5][:-50])
print("N(nu_obs)   =", len(nu_obs))


## 3. Bundle, PT profile, and chemistry backends

* **Emulator** - JIT, gradient-traceable, used in NUTS.
* **Live FastChem** - subprocess, used to generate truth mocks and as the
  ground-truth reference at every truth point.
* **ExoGibbs (matched-thermo)** - built via `chemsetup_matched_to_fastchem`
  so its species and thermo coefficients match FastChems `logK_wo_ions.dat`.
  Used **only as a forward-VMR overlay**, never inside a chain.

Hard sanity gate: `bundle.fixed_globals == {}`. A non-empty value means a
global input was held constant during training and gradient-based sampling
on that direction is structurally zero.

In [ ]:
from src.constants import SOLAR_ABUNDANCES
from src.models.standalone_inference import (
    guillot_temperature,
    load_model,
    make_fastchem_vmr_fn,
)
from src.models.classical_reference import (
    build_exogibbs_element_vector,
    build_exogibbs_species_indices,
    chemsetup_matched_to_fastchem,
    mean_abs_log10_error,
    resolve_vulcan_source_root,
    run_fastchem_online,
)
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile

bundle = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")
FASTCHEM_SOURCE_ROOT = resolve_vulcan_source_root(bundle.config, project_root=PROJECT_ROOT)

assert bundle.fixed_globals == {}, (
    f"bundle.fixed_globals = {bundle.fixed_globals!r}; gradients on a held-fixed "
    "global are structurally zero. Re-export from a training run with no fixed globals."
)

chem = chemsetup_matched_to_fastchem(FASTCHEM_SOURCE_ROOT)
EG_IDX = build_exogibbs_species_indices(chem, species_labels)
EG_OPTS = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")

print(f"chemistry      : {bundle.chemistry_type}  | model: {bundle.model_type}")
print(f"output species : {species_labels}")
print(f"global order   : {bundle.data_contract['global_static_feature_order']}")
print(f"fixed_globals  : {bundle.fixed_globals}")
print(f"num_levels rng : {tuple(bundle.data_contract['num_levels_range'])}")
print(f"P union range  : 10**{bundle.data_contract['log10_pressure_bar_union_range']} bar")
print(f"ExoGibbs species: {len(chem.species)}  | elements: {len(chem.elements)}")

IDX_CO = species_labels.index("CO")
IDX_H2 = species_labels.index("H2")
IDX_H2O = species_labels.index("H2O")


## 4. PT profile and `logZ` parameterization

PT profile is the Guillot 2010 / Piette+2019 form (training family). In the
forward model and inside NUTS we expose ``T_int_k``, ``T_eq_k``, and
``log10_gamma`` as free parameters; ``log10_delta`` stays pinned (it is
absorbed into the per-target ``logg`` because both control the IR
optical-depth scale).

`logZ` retrieves a single metallicity scaling that multiplies **C and O
simultaneously** by `10**logZ` around the solar anchor. He, N, S stay at
solar.

Solar anchor: `src.constants.SOLAR_ABUNDANCES` (Asplund 2009 - the *training*
anchor). The OG used AAG21 via `exojax.utils.zsol.nsol()`; feeding the
emulator AAG21 instead would offset the truth point by a few percent per
element and is off-contract per `spec.md`.

In [ ]:
LOG10_DELTA_PIN = 0.0  # delta = 1 bar^-1; kappa_IR/g, absorbed into logg.

LOG_SOLAR = {f"log_{k[:-2]}_H": float(np.log10(v)) for k, v in SOLAR_ABUNDANCES.items()}

def globals_from_logZ_np(logZ):
    s = float(10.0 ** logZ)
    return {
        "He_H": float(SOLAR_ABUNDANCES["He_H"]),
        "C_H":  float(SOLAR_ABUNDANCES["C_H"]) * s,
        "O_H":  float(SOLAR_ABUNDANCES["O_H"]) * s,
        "N_H":  float(SOLAR_ABUNDANCES["N_H"]),
        "S_H":  float(SOLAR_ABUNDANCES["S_H"]),
    }

def globals_from_logZ_jax(logZ):
    s = 10.0 ** logZ
    return {
        "He_H": jnp.asarray(SOLAR_ABUNDANCES["He_H"], dtype=jnp.float64),
        "C_H":  jnp.asarray(SOLAR_ABUNDANCES["C_H"], dtype=jnp.float64) * s,
        "O_H":  jnp.asarray(SOLAR_ABUNDANCES["O_H"], dtype=jnp.float64) * s,
        "N_H":  jnp.asarray(SOLAR_ABUNDANCES["N_H"], dtype=jnp.float64),
        "S_H":  jnp.asarray(SOLAR_ABUNDANCES["S_H"], dtype=jnp.float64),
    }

print("solar (training anchor) X/H:")
for k, v in SOLAR_ABUNDANCES.items():
    print(f"  {k:<5s} = {v:.4e}  (log10 = {np.log10(v):+.4f})")
print(f"\\nLOG10_DELTA_PIN = {LOG10_DELTA_PIN}")


def Tarr_guillot(t_int_k, t_eq_k, log10_gamma):
    """JAX-traceable Guillot (alpha=0) profile on art.pressure."""
    return guillot_temperature(
        art.pressure,
        t_int_k=t_int_k,
        t_eq_k=t_eq_k,
        log10_delta=LOG10_DELTA_PIN,
        log10_gamma=log10_gamma,
    )


def Tarr_guillot_np(t_int_k, t_eq_k, log10_gamma):
    return np.asarray(Tarr_guillot(float(t_int_k), float(t_eq_k), float(log10_gamma)))


## 5. Three VMR backends on `(Tarr, art.pressure, logZ)`

In [ ]:
@jax.jit
def vmr_emulator(Tarr, logZ):
    # Emulator -> (NLAYER, 17) VMR table aligned to species_labels. JAX-traceable.
    return vmr_fn(Tarr, art.pressure, globals_from_logZ_jax(logZ))


def vmr_fastchem_call(Tarr, logZ, return_fail_mask=False):
    # Live FastChem rerun (subprocess). Forward-only; never inside NUTS.
    return run_fastchem_online(
        FASTCHEM_SOURCE_ROOT,
        np.asarray(art.pressure),
        np.asarray(Tarr),
        globals_from_logZ_np(float(logZ)),
        species_labels,
        bundle.config,
        return_fail_mask=return_fail_mask,
    )


def vmr_exogibbs_call(Tarr, logZ):
    # ExoGibbs -> (NLAYER, 17) VMR table. Used only as a forward overlay.
    ev = build_exogibbs_element_vector(
        chem,
        globals_from_logZ_np(float(logZ)),
        mode="fastchem_proxy",
    )
    res = equilibrium_profile(
        chem,
        np.asarray(Tarr),
        np.asarray(art.pressure),
        ev,
        Pref=1.0,
        options=EG_OPTS,
    )
    return np.asarray(res.x[:, EG_IDX])


## 6. Forward spectrum (CO + H2-H2 CIA, OG-style)

Same opacity stack as the OG (CO line + CIA continuum). Mean molecular weight
is computed **per layer** from the full 17-species emulator VMR profile,
rather than pinned at 2.33 like the OG (the OG flagged the constant-mmw
shortcut as "not accurate"; this fix means the continuum responds to
abundance changes).

Two `fspec(...)` functions, identical downstream of the chemistry layer:

* `fspec_emulator(...)` - JAX-jit, used in NUTS and MAP.
* `fspec_fastchem(...)` - numpy-only, used to generate truth mocks and to
  evaluate "what would FastChem have predicted at this MAP point?"

Both take the Guillot params ``(t_int, t_eq, log_gamma)`` instead of
``(T0, alpha)``.

In [ ]:
from exojax.atm.atmconvert import vmr_to_mmr
from exojax.database.molinfo.mass import isotope_molmass

MOLMASS_CO = isotope_molmass("12C-16O")

# Per-species molar masses (g/mol), positionally matched to species_labels.
MOLAR_MASS = {"H2": 2.016, "He": 4.003, "H": 1.008, "O": 15.999, "OH": 17.007,
              "H2O": 18.015, "CO": 28.010, "CO2": 44.009, "CH4": 16.043, "N2": 28.014,
              "NH3": 17.031, "H2S": 34.081, "SH": 33.073, "S": 32.065, "SO": 48.064,
              "SO2": 64.064, "S2": 64.130}
MASS_VEC = jnp.array([MOLAR_MASS[s] for s in species_labels], dtype=jnp.float64)


def _spec_core(Tarr, gravity_cgs, RV, vsini, vmr_table):
    # Shared RT stack: CO line + H2-H2 CIA + rotation + IP + sampling.
    vmr_co = vmr_table[:, IDX_CO]
    vmr_h2 = vmr_table[:, IDX_H2]
    mmw = jnp.sum(vmr_table * MASS_VEC, axis=-1)

    mmr_co = vmr_to_mmr(vmr_co, MOLMASS_CO, mmw)

    xs_co = opa.xsmatrix(Tarr, art.pressure)
    dtau_co = art.opacity_profile_xs(xs_co, mmr_co, MOLMASS_CO, gravity_cgs)

    logacia = opacia.logacia_matrix(Tarr)
    dtau_cia = art.opacity_profile_cia(
        logacia, Tarr, vmr_h2, vmr_h2, mmw[:, None], gravity_cgs
    )

    F = art.run(dtau_co + dtau_cia, Tarr)
    F = sop_rot.rigid_rotation(F, vsini, U1, U2)
    F = sop_inst.ipgauss(F, BETA_INST)
    return sop_inst.sampling(F, RV, nu_obs)


@jax.jit
def fspec_emulator(t_int, t_eq, log_gamma, logg, RV, vsini, logZ):
    Tarr = Tarr_guillot(t_int, t_eq, log_gamma)
    vmr = vmr_emulator(Tarr, logZ)
    return _spec_core(Tarr, 10.0 ** logg, RV, vsini, vmr)


def fspec_fastchem(t_int, t_eq, log_gamma, logg, RV, vsini, logZ):
    # Forward-only (subprocess); not differentiable. Numpy returned.
    Tarr = Tarr_guillot_np(t_int, t_eq, log_gamma)
    vmr = jnp.asarray(vmr_fastchem_call(Tarr, float(logZ)),
                      dtype=jnp.float64)
    return np.asarray(_spec_core(jnp.asarray(Tarr), 10.0 ** logg, RV, vsini, vmr))


from exojax.utils.astrofunc import gravity_jupiter
LOGG_TRUTH = float(np.log10(gravity_jupiter(1.0, 10.0)))  # ~4.39 (cgs)

# Warmup compile.
_warm = fspec_emulator(400.0, 1500.0, -1.0, LOGG_TRUTH, 40.0, 10.0, 0.0)
_warm.block_until_ready()
print(f"fspec_emulator first-call OK; output shape = {tuple(_warm.shape)}")
print(f"LOGG_TRUTH (Jupiter, 10 Mj) = {LOGG_TRUTH:.4f} log10(cm/s^2)")


## 7. Truth grid: full prior coverage on `logZ`

Five truths bracket the OGs `logZ ~ U(-1, 1)` prior plus one off-axis
shift in equilibrium temperature. Each truth produces an independent
FastChem mock; recovery on every truth is a separate NUTS run.

The PT profile params come from the bundles training distribution
(Piette+2019 / Guillot 2010 family with ``alpha=0``):

| name           | logZ | T_int | T_eq | log10_gamma | comment |
|----------------|------|-------|------|-------------|---------|
| `logZ_-0.5`    | -0.5 | 400 K | 1500 K | -1.0      | sub-solar |
| `logZ_0.0`     |  0.0 | 400 K | 1500 K | -1.0      | solar |
| `logZ_+0.5`    | +0.5 | 400 K | 1500 K | -1.0      | super-solar mid |
| `logZ_+0.7`    | +0.7 | 400 K | 1500 K | -1.0      | super-solar (edge of training) |
| `cool_solar`   |  0.0 | 400 K | 1300 K | -1.0      | cooler equilibrium temperature |

These produce profiles peaking ~1500-1700 K near the photosphere, dropping
to ~1300 K aloft - the regime the bundle saw most heavily during
training.

In [ ]:
TRUTHS = [
    {"name": "logZ_-0.5",  "t_int": 400.0, "t_eq": 1500.0, "log_gamma": -1.0,
     "logg": LOGG_TRUTH, "RV": 40.0, "vsini": 10.0, "logZ": -0.5},
    {"name": "logZ_0.0",   "t_int": 400.0, "t_eq": 1500.0, "log_gamma": -1.0,
     "logg": LOGG_TRUTH, "RV": 40.0, "vsini": 10.0, "logZ":  0.0},
    {"name": "logZ_+0.5",  "t_int": 400.0, "t_eq": 1500.0, "log_gamma": -1.0,
     "logg": LOGG_TRUTH, "RV": 40.0, "vsini": 10.0, "logZ": +0.5},
    {"name": "logZ_+0.7",  "t_int": 400.0, "t_eq": 1500.0, "log_gamma": -1.0,
     "logg": LOGG_TRUTH, "RV": 40.0, "vsini": 10.0, "logZ": +0.7},
    {"name": "cool_solar", "t_int": 400.0, "t_eq": 1300.0, "log_gamma": -1.0,
     "logg": LOGG_TRUTH, "RV": 40.0, "vsini": 10.0, "logZ":  0.0},
]
PARAMS_KW = ("t_int", "t_eq", "log_gamma", "logg", "RV", "vsini", "logZ")

for T in TRUTHS:
    Tprof = Tarr_guillot_np(T["t_int"], T["t_eq"], T["log_gamma"])
    print(f"  {T['name']:<12s}  logZ={T['logZ']:+.2f}  T_int={T['t_int']:.0f}K  "
          f"T_eq={T['t_eq']:.0f}K  log_gamma={T['log_gamma']:+.2f}  "
          f"Tarr=[{Tprof.min():.0f}, {Tprof.max():.0f}]K")


## 8. Forward VMR fidelity at every truth

The bundle has known multi-dex disagreement on trace radicals (`O`, `OH`,
`H`, `S`, `S2`, `SO`, `SO2`) which the CO + H2-H2 CIA forward model never
reads. So an all-species `mean |dex log10 VMR|` is the wrong shape for a
replacement test - it averages the irrelevant tail with the relevant
species (`CO`, `H2`, `H2O`).

This cell prints both the all-species and per-species (CO, H2, H2O) MAEs
plus the FastChem-vs-ExoGibbs residual. The hard replacement gate moves
to cell 23 (mock generation), where we check
`rms(mu_emulator - mu_fastchem) / noise <= 1.0` at every truth - the
observationally meaningful gate.

In [ ]:
SPEC_RELEVANT = ("CO", "H2", "H2O")
_eps = 1e-30


def _per_species_mae(a, b):
    la = np.log10(np.clip(np.asarray(a), _eps, None))
    lb = np.log10(np.clip(np.asarray(b), _eps, None))
    return np.mean(np.abs(la - lb), axis=0)  # (n_species,)


TRUTH_VMR = {}
header = (f"{'truth':<14s}  {'all|ML-FC|':>11s}  " +
          "  ".join(f"{sp+'(ML-FC)':>11s}" for sp in SPEC_RELEVANT) +
          f"  {'FC-EG':>9s}  {'fc_fail':>8s}")
print(header)
print("-" * len(header))
for T in TRUTHS:
    Tarr = Tarr_guillot(T["t_int"], T["t_eq"], T["log_gamma"])
    vm = np.asarray(vmr_emulator(Tarr, T["logZ"]))
    vf, vf_fail = vmr_fastchem_call(np.asarray(Tarr), T["logZ"], return_fail_mask=True)
    vf = np.asarray(vf)
    ve = vmr_exogibbs_call(np.asarray(Tarr), T["logZ"])

    e_mf_all = mean_abs_log10_error(vm, vf)
    e_fe_all = mean_abs_log10_error(vf, ve)
    e_me_all = mean_abs_log10_error(vm, ve)
    per_sp_mf = _per_species_mae(vm, vf)
    per_sp_fe = _per_species_mae(vf, ve)
    per_sp_me = _per_species_mae(vm, ve)
    sp_rel_mf = {sp: float(per_sp_mf[species_labels.index(sp)]) for sp in SPEC_RELEVANT}
    sp_rel_fe = {sp: float(per_sp_fe[species_labels.index(sp)]) for sp in SPEC_RELEVANT}
    sp_rel_me = {sp: float(per_sp_me[species_labels.index(sp)]) for sp in SPEC_RELEVANT}
    ff = int(np.asarray(vf_fail).sum()) if vf_fail is not None else 0

    TRUTH_VMR[T["name"]] = dict(
        Tarr=np.asarray(Tarr), ml=vm, fc=vf, eg=ve, fc_fail=ff,
        ml_fc=e_mf_all, fc_eg=e_fe_all, ml_eg=e_me_all,
        per_sp_ml_fc=sp_rel_mf,
        per_sp_fc_eg=sp_rel_fe,
        per_sp_ml_eg=sp_rel_me,
        per_sp_ml_fc_full={sp: float(per_sp_mf[i]) for i, sp in enumerate(species_labels)},
    )
    sp_cols = "  ".join(f"{sp_rel_mf[sp]:>11.4f}" for sp in SPEC_RELEVANT)
    print(f"{T['name']:<14s}  {e_mf_all:>11.4f}  {sp_cols}  {e_fe_all:>9.4f}  {ff:>8d}")

print()
print("Notes:")
print("  - 'all|ML-FC|' averages over all 17 species (incl. trace radicals).")
print("  - Spectrum-relevant species (CO, H2, H2O) are what the CO+CIA forward")
print("    model actually reads. Trace-radical disagreement is expected and")
print("    does not invalidate the replacement test.")
print("  - Hard gate is at the spectrum level (next cell, mock RMS / noise).")


In [ ]:
# Per-species, per-truth VMR overlay for the four most diagnostic species.
fig, axes = plt.subplots(len(TRUTHS), 4, figsize=(16, 3.4 * len(TRUTHS)),
                         sharey=True)
if len(TRUTHS) == 1:
    axes = axes[None, :]

for row, T in enumerate(TRUTHS):
    rec = TRUTH_VMR[T["name"]]
    for col, sp in enumerate(("CO", "H2O", "H2", "CH4")):
        ax = axes[row, col]
        i = species_labels.index(sp)
        ax.plot(rec["fc"][:, i], art.pressure, lw=1.4, label="FastChem (truth)")
        ax.plot(rec["ml"][:, i], art.pressure, lw=1.0, ls="--", label="emulator")
        ax.plot(rec["eg"][:, i], art.pressure, lw=1.0, ls=":", label="ExoGibbs")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.invert_yaxis()
        if col == 0:
            ax.set_ylabel(f"P (bar)\n[{T['name']}]")
        if row == 0:
            ax.set_title(sp)
        if row == len(TRUTHS) - 1:
            ax.set_xlabel(f"VMR ({sp})")
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 9. Mock spectra generated by **live FastChem**

One mock per truth, generated by the live-FastChem subprocess. The
emulator-at-truth prediction is overlaid (no chain involvement) so the
chemistry residual `mu_emulator - mu_fastchem` is visible.

**Diagnostic, not a gate.** When `rms(mu_ml - mu_fc) / noise > 1` the
chemistry residual exceeds the data noise floor at that truth - but the
chain has `sigmain` as a free parameter, so it will absorb a portion of
that residual into an inflated noise estimate. The actual replacement
test is whether the `logZ` posterior recovers the truth unbiased
*despite* the residual; bias in `logZ_bias_vs_truth` is what fails the
claim, not residual size alone.

In [ ]:
NOISE_TRUE = 500.0  # erg/s/cm^2/cm^-1, matches OG.

rng = default_rng(seed=20260430)
MOCKS = {}
_warn_truths = []

for T in TRUTHS:
    name = T["name"]
    mu_fc = fspec_fastchem(*[T[k] for k in PARAMS_KW])
    mu_ml = np.asarray(fspec_emulator(*[T[k] for k in PARAMS_KW]))
    Fobs = mu_fc + rng.normal(0.0, NOISE_TRUE, size=len(nu_obs))
    rms_ml_fc = float(np.sqrt(np.mean((mu_ml - mu_fc) ** 2)))
    rms_over_noise = rms_ml_fc / NOISE_TRUE
    MOCKS[name] = dict(
        truth=T, mu_fc=mu_fc, mu_ml=mu_ml, Fobs=Fobs,
        spec_rms_ml_fc=rms_ml_fc,
        spec_rms_ml_fc_over_noise=rms_over_noise,
    )
    flag = "OK" if rms_over_noise <= 1.0 else "WARN"
    print(f"[{name:<12s}] mock RMS(ML-FC) at truth = {rms_ml_fc:7.2f}  "
          f"({rms_over_noise:5.3f} x noise)  [{flag}]")
    if rms_over_noise > 1.0:
        _warn_truths.append((name, rms_over_noise))

print()
if _warn_truths:
    print(f"!! {len(_warn_truths)}/{len(TRUTHS)} truths have chemistry residual > noise:")
    for n, r in _warn_truths:
        print(f"     {n}: rms/n = {r:.2f}")
    print("   The chain will absorb part of this into `sigmain`. The actual")
    print("   replacement test is `logZ_bias_vs_truth` in the headline at the end.")
else:
    print("All truths have chemistry residual within the data noise floor.")
print()

# Visualization.
n_truths = len(TRUTHS)
ncols = min(n_truths, 3)
nrows = int(np.ceil(n_truths / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.4 * nrows),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes).reshape(nrows, ncols)
for idx, T in enumerate(TRUTHS):
    ax = axes[idx // ncols, idx % ncols]
    name = T["name"]
    M = MOCKS[name]
    ax.plot(nu_obs, M["mu_fc"], lw=1.0, label="FastChem (truth)")
    ax.plot(nu_obs, M["mu_ml"], lw=0.9, ls="--", label="emulator @ truth")
    ax.errorbar(nu_obs, M["Fobs"], NOISE_TRUE, fmt=".", color="gray", alpha=0.35,
                label="mock data")
    ax.set_title(f"{name}  (logZ={T['logZ']:+.2f}, "
                 f"rms/n={M['spec_rms_ml_fc_over_noise']:.2f})")
    ax.legend(fontsize=7)
for k in range(n_truths, nrows * ncols):
    axes[k // ncols, k % ncols].set_visible(False)
fig.supxlabel("wavenumber (cm$^{-1}$)")
fig.supylabel("flux (erg/s/cm$^2$/cm$^{-1}$)")
plt.tight_layout()
plt.show()


## 10. NUTS retrieval - emulator backend, Guillot priors

Same retrieval shape as the OG (CO + H2-H2 CIA, single ``logZ`` scaling on
C/O), but with the Guillot PT profile:

| param        | prior |
|--------------|-------|
| `t_int`      | `Uniform(200, 800)` K |
| `t_eq`       | `Uniform(1000, 2000)` K |
| `log_gamma`  | `Uniform(-1.5, 0.5)` |
| `logg`       | `Uniform(4.0, 5.0)` |
| `RV`         | `Uniform(35, 45)` |
| `vsini`      | `Uniform(5, 15)` |
| `logZ`       | `Uniform(-1.0, +1.0)` |
| `sigmain`    | `Exponential(1e-3)` |

500 warmup + 1000 samples per truth, single chain, `target_accept_prob=0.8`,
`max_tree_depth=8`. Every chain runs against a **FastChem-generated** mock,
so a recovered posterior centered on the truth means the emulator faithfully
replaced FastChem inside the kernel.

In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random


def _make_model():
    def model(spectrum):
        t_int   = numpyro.sample("t_int",   dist.Uniform(200.0, 800.0))
        t_eq    = numpyro.sample("t_eq",    dist.Uniform(1000.0, 2000.0))
        log_gamma = numpyro.sample("log_gamma", dist.Uniform(-1.5, 0.5))
        logg    = numpyro.sample("logg",    dist.Uniform(4.0, 5.0))
        RV      = numpyro.sample("RV",      dist.Uniform(35.0, 45.0))
        vsini   = numpyro.sample("vsini",   dist.Uniform(5.0, 15.0))
        logZ    = numpyro.sample("logZ",    dist.Uniform(-1.0, 1.0))
        sigmain = numpyro.sample("sigmain", dist.Exponential(1.0e-3))
        mu = fspec_emulator(t_int, t_eq, log_gamma, logg, RV, vsini, logZ)
        numpyro.sample("spectrum", dist.Normal(mu, sigmain), obs=spectrum)
    return model


NUM_WARMUP, NUM_SAMPLES = 500, 1000
MAX_TREE = 8
SAMPLE_PARAMS = ("t_int", "t_eq", "log_gamma", "logg", "RV", "vsini", "logZ", "sigmain")

CHAINS = {}
for T in TRUTHS:
    name = T["name"]
    print(f"\n=== {name}: NUTS warmup={NUM_WARMUP} samples={NUM_SAMPLES} ===")
    kernel = NUTS(
        _make_model(),
        forward_mode_differentiation=False,
        max_tree_depth=MAX_TREE,
        target_accept_prob=0.8,
    )
    mcmc = MCMC(
        kernel,
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        num_chains=1,
        progress_bar=True,
    )
    seed = int(np.random.default_rng().integers(0, 2 ** 31 - 1))
    t0 = time.perf_counter()
    mcmc.run(
        random.PRNGKey(seed),
        spectrum=jnp.asarray(MOCKS[name]["Fobs"]),
        extra_fields=("num_steps", "diverging", "accept_prob",
                      "mean_accept_prob", "energy", "potential_energy"),
    )
    dt = time.perf_counter() - t0
    n_iter = NUM_WARMUP + NUM_SAMPLES
    print(f"  wall: {dt:.1f} s  ({dt / n_iter * 1e3:.0f} ms/iter)")
    mcmc.print_summary()
    CHAINS[name] = dict(
        mcmc=mcmc,
        samples=mcmc.get_samples(),
        extra=mcmc.get_extra_fields(),
        wall_s=dt,
        seed=seed,
    )


## 11. Recovery summary

For each truth: posterior mean +/- std, bias vs truth (in dex for `logZ`,
in K for the PT params), `z_vs_truth` (bias / posterior std). The
replacement claim wants:

* `|bias_vs_truth|` <= ~0.5 sigma per parameter (especially `logZ`).
* `n_diverging == 0` (no chain pathology).
* spectrum residual at MAP < ~1 x noise.

In [ ]:
def _ess_single_chain(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    y = x - x.mean()
    var0 = float((y * y).mean())
    if var0 == 0.0 or n < 4:
        return float(n)
    max_lag = min(n - 1, 200)
    acf = np.array([(y[: n - k] * y[k:]).mean() / var0 for k in range(1, max_lag + 1)])
    cut = np.where(acf <= 0.0)[0]
    end = int(cut[0]) if cut.size else len(acf)
    tau = 1.0 + 2.0 * float(acf[:end].sum())
    return float(n / max(tau, 1.0))


def _summary_row(samples, T):
    rows = []
    for p in SAMPLE_PARAMS:
        x = np.asarray(samples[p])
        m, s = float(x.mean()), float(x.std())
        truth = T.get(p, np.nan)
        bias = m - truth if np.isfinite(truth) else np.nan
        z = bias / max(s, 1e-12) if np.isfinite(bias) else np.nan
        rows.append((p, truth, m, s, bias, z, _ess_single_chain(x)))
    return rows

print(f"{'truth':<12}  {'param':<10}{'truth':>12}{'mean':>12}{'std':>11}"
      f"{'bias':>12}{'z':>8}{'ess':>9}")
print("-" * 86)
for T in TRUTHS:
    name = T["name"]
    rows = _summary_row(CHAINS[name]["samples"], T)
    for p, tv, m, s, b, z, e in rows:
        tv_str = f"{tv:>12.4f}" if np.isfinite(tv) else f"{'(--)':>12s}"
        b_str = f"{b:>+12.4f}" if np.isfinite(b) else f"{'(--)':>12s}"
        z_str = f"{z:>+8.2f}" if np.isfinite(z) else f"{'(--)':>8s}"
        print(f"{name:<12}  {p:<10}{tv_str}{m:>12.4f}{s:>11.4f}{b_str}{z_str}{e:>9.0f}")
    print()


In [ ]:
# Spectrum residuals at posterior mean and at MAP via scipy.
from scipy.optimize import minimize

def _nll_factory(Fobs_j):
    @jax.jit
    def nll(x):
        mu = fspec_emulator(x[0], x[1], x[2], x[3], x[4], x[5], x[6])
        sig = jnp.exp(x[7])  # sigmain in log-space for the optimizer.
        return 0.5 * jnp.sum(((mu - Fobs_j) / sig) ** 2 + 2.0 * jnp.log(sig))

    return nll, jax.jit(jax.grad(nll))


BOUNDS = [(200.0, 800.0), (1000.0, 2000.0), (-1.5, 0.5),
          (4.0, 5.0), (35.0, 45.0), (5.0, 15.0), (-1.0, 1.0),
          (np.log(50.0), np.log(5000.0))]

print(f"{'truth':<12}  {'rms_ml@truth':>14}{'rms_ml@map':>13}"
      f"{'rms_fc@map':>13}{'rms/noise':>11}")
print("-" * 70)
SPEC_AT_MAP = {}
for T in TRUTHS:
    name = T["name"]
    Fobs = MOCKS[name]["Fobs"]
    Fobs_j = jnp.asarray(Fobs)

    nll, gnll = _nll_factory(Fobs_j)
    s = CHAINS[name]["samples"]
    x0 = np.array([float(np.mean(s[p])) for p in SAMPLE_PARAMS[:-1]] +
                  [np.log(float(np.mean(s["sigmain"])))])
    res = minimize(
        lambda x: float(nll(jnp.asarray(x))),
        x0,
        jac=lambda x: np.asarray(gnll(jnp.asarray(x)), dtype=np.float64),
        method="L-BFGS-B",
        bounds=BOUNDS,
        options=dict(maxiter=200, ftol=1e-10, gtol=1e-8),
    )
    x_map = res.x
    mu_ml_at_map = np.asarray(fspec_emulator(*x_map[:7]))
    mu_fc_at_map = fspec_fastchem(*x_map[:7])
    mu_ml_truth = MOCKS[name]["mu_ml"]
    rms_ml_truth = float(np.sqrt(np.mean((mu_ml_truth - Fobs) ** 2)))
    rms_ml_map = float(np.sqrt(np.mean((mu_ml_at_map - Fobs) ** 2)))
    rms_fc_map = float(np.sqrt(np.mean((mu_fc_at_map - Fobs) ** 2)))

    print(f"{name:<12}  {rms_ml_truth:>14.2f}{rms_ml_map:>13.2f}"
          f"{rms_fc_map:>13.2f}{rms_ml_map / NOISE_TRUE:>11.3f}")

    SPEC_AT_MAP[name] = dict(
        x_map=x_map,
        mu_ml_at_map=mu_ml_at_map,
        mu_fc_at_map=mu_fc_at_map,
        rms_ml_truth=rms_ml_truth,
        rms_ml_map=rms_ml_map,
        rms_fc_map=rms_fc_map,
        nll_star=float(res.fun),
        nfev=int(res.nfev),
        success=bool(res.success),
    )


## 12. Headline plots

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(TRUTHS)))
for c, T in zip(colors, TRUTHS):
    name = T["name"]
    s = np.asarray(CHAINS[name]["samples"]["logZ"])
    ax.hist(s, bins=30, density=True, color=c, alpha=0.30, edgecolor="none",
            label=f"{name} (truth={T['logZ']:+.2f})")
    ax.axvline(T["logZ"], color=c, ls=":", lw=1.4)
ax.set_xlabel("logZ (dex)")
ax.set_ylabel("posterior density")
ax.set_title("logZ recovery across the prior  -  prior is U(-1, +1)")
ax.set_xlim(-1.0, 1.0)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import gaussian_kde

CORNER_PARAMS = ("t_int", "t_eq", "log_gamma", "logg", "RV", "vsini", "logZ", "sigmain")
PRIOR_BOUNDS = {
    "t_int": (200.0, 800.0), "t_eq": (1000.0, 2000.0), "log_gamma": (-1.5, 0.5),
    "logg": (4.0, 5.0), "RV": (35.0, 45.0), "vsini": (5.0, 15.0),
    "logZ": (-1.0, 1.0), "sigmain": (50.0, 2000.0),
}


def _corner_one(samples, truth, name, bounds=None, bins=25, figsize=(11, 11)):
    n = len(CORNER_PARAMS)
    fig, axes = plt.subplots(n, n, figsize=figsize)
    arr = np.stack([np.asarray(samples[p]) for p in CORNER_PARAMS], axis=1)
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
                continue
            xi = arr[:, i]
            xj = arr[:, j]
            if i == j:
                ax.hist(xi, bins=bins, density=True, color="C0", alpha=0.30,
                        edgecolor="none")
                try:
                    kde = gaussian_kde(xi)
                    xs = np.linspace(xi.min(), xi.max(), 200)
                    ax.plot(xs, kde(xs), color="C0", lw=1.4)
                except np.linalg.LinAlgError:
                    pass
                if CORNER_PARAMS[i] in truth:
                    ax.axvline(truth[CORNER_PARAMS[i]], color="k", ls=":", lw=1.2)
                ax.set_yticks([])
            else:
                ax.scatter(xj, xi, s=4, color="C0", alpha=0.25, edgecolor="none")
                try:
                    kde = gaussian_kde(np.vstack([xj, xi]))
                    xx, yy = np.meshgrid(np.linspace(xj.min(), xj.max(), 40),
                                         np.linspace(xi.min(), xi.max(), 40))
                    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
                    ax.contour(xx, yy, zz, levels=3, colors="C0",
                               linewidths=0.7, alpha=0.85)
                except (np.linalg.LinAlgError, ValueError):
                    pass
                if CORNER_PARAMS[j] in truth:
                    ax.axvline(truth[CORNER_PARAMS[j]], color="k", ls=":", lw=0.9)
                if CORNER_PARAMS[i] in truth:
                    ax.axhline(truth[CORNER_PARAMS[i]], color="k", ls=":", lw=0.9)
            if bounds is not None:
                if CORNER_PARAMS[j] in bounds:
                    ax.set_xlim(*bounds[CORNER_PARAMS[j]])
                if i != j and CORNER_PARAMS[i] in bounds:
                    ax.set_ylim(*bounds[CORNER_PARAMS[i]])
            if i < n - 1:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel(CORNER_PARAMS[j], fontsize=8)
                ax.tick_params(axis="x", labelsize=7)
            if j > 0 or i == 0:
                ax.set_yticklabels([])
            else:
                ax.set_ylabel(CORNER_PARAMS[i], fontsize=8)
                ax.tick_params(axis="y", labelsize=7)
    fig.suptitle(f"{name}  -  truth logZ={truth.get('logZ', np.nan):+.2f}",
                 fontsize=10, y=0.995)
    fig.align_labels()
    plt.tight_layout()
    return fig


for T in TRUTHS:
    _corner_one(CHAINS[T["name"]]["samples"], T, T["name"], bounds=PRIOR_BOUNDS)
    plt.show()


## 13. Diagnostics dump

In [ ]:
import json
import datetime

OUTDIR = (
    Path("diagnostics")
    / ("recovery_" + datetime.datetime.now(datetime.timezone.utc)
       .strftime("%Y%m%dT%H%M%SZ"))
)
OUTDIR.mkdir(parents=True, exist_ok=True)
print("writing diagnostics to:", OUTDIR.resolve())


def _pstats(x):
    x = np.asarray(x)
    q = np.quantile(x, [0.025, 0.16, 0.5, 0.84, 0.975])
    return {
        "mean": float(x.mean()), "std": float(x.std()), "median": float(q[2]),
        "q16": float(q[1]), "q84": float(q[3]),
        "q025": float(q[0]), "q975": float(q[4]),
        "n": int(x.size),
    }


def _ext_summary(ex):
    out = {}
    if not ex:
        return out
    if "diverging" in ex:
        d = np.asarray(ex["diverging"])
        out["n_diverging"] = int(d.sum())
        out["frac_diverging"] = float(d.mean())
    if "num_steps" in ex:
        ns = np.asarray(ex["num_steps"])
        out["num_steps"] = {
            "mean": float(ns.mean()),
            "median": float(np.median(ns)),
            "max": int(ns.max()),
            "p99": float(np.quantile(ns, 0.99)),
        }
    if "accept_prob" in ex:
        ap = np.asarray(ex["accept_prob"])
        out["accept_prob"] = {
            "mean": float(ap.mean()),
            "min": float(ap.min()),
        }
    if "energy" in ex:
        out["energy_std"] = float(np.asarray(ex["energy"]).std())
    return out


per_truth = {}
posteriors_npz = {}
vmr_npz = {
    "pressure_bar": np.asarray(art.pressure),
    "species_labels": np.array(species_labels),
    "nu_obs": np.asarray(nu_obs),
}

for T in TRUTHS:
    name = T["name"]
    samples = CHAINS[name]["samples"]
    extra = CHAINS[name]["extra"]
    rec_vmr = TRUTH_VMR[name]
    spec = SPEC_AT_MAP[name]

    stats = {}
    for p in SAMPLE_PARAMS:
        x = np.asarray(samples[p])
        stats[p] = {
            **_pstats(x),
            "truth": float(T[p]) if p in T else None,
            "bias_vs_truth": float(x.mean() - T[p]) if p in T else None,
            "z_vs_truth": (
                float((x.mean() - T[p]) / max(x.std(), 1e-12)) if p in T else None
            ),
        }

    per_truth[name] = {
        "truth": {p: float(T[p]) for p in PARAMS_KW},
        "wall_s_nuts": float(CHAINS[name]["wall_s"]),
        "n_samples": int(NUM_SAMPLES),
        "n_warmup": int(NUM_WARMUP),
        "max_tree_depth": int(MAX_TREE),
        "seed": int(CHAINS[name]["seed"]),
        "posterior_stats": stats,
        "nuts_extras_summary": _ext_summary(extra),
        "fastchem_monitor_fail_rows_at_truth": int(rec_vmr["fc_fail"]),
        "vmr_mean_abs_log10_at_truth": {
            "ml_vs_fastchem": float(rec_vmr["ml_fc"]),
            "fastchem_vs_exogibbs": float(rec_vmr["fc_eg"]),
            "ml_vs_exogibbs": float(rec_vmr["ml_eg"]),
        },
        "spectrum_rms_at_truth": {
            "ml_vs_fastchem": float(np.sqrt(np.mean(
                (MOCKS[name]["mu_ml"] - MOCKS[name]["mu_fc"]) ** 2
            ))),
            "ml_vs_fastchem_over_noise": float(np.sqrt(np.mean(
                (MOCKS[name]["mu_ml"] - MOCKS[name]["mu_fc"]) ** 2
            )) / NOISE_TRUE),
        },
        "spectrum_rms_at_map": {
            "ml_vs_obs": float(spec["rms_ml_map"]),
            "fc_ml_params_vs_obs": float(spec["rms_fc_map"]),
            "ml_vs_obs_over_noise": float(spec["rms_ml_map"] / NOISE_TRUE),
        },
        "map_x": {p: float(spec["x_map"][i]) for i, p in enumerate(SAMPLE_PARAMS[:-1])} | {
            "sigmain": float(np.exp(spec["x_map"][-1])),
        },
    }

    for p, x in samples.items():
        posteriors_npz[f"{name}/{p}"] = np.asarray(x)
    vmr_npz[f"{name}/Tarr"] = rec_vmr["Tarr"]
    vmr_npz[f"{name}/emulator"] = rec_vmr["ml"]
    vmr_npz[f"{name}/fastchem"] = rec_vmr["fc"]
    vmr_npz[f"{name}/exogibbs"] = rec_vmr["eg"]
    vmr_npz[f"{name}/Fobs"] = MOCKS[name]["Fobs"]
    vmr_npz[f"{name}/mu_fc_truth"] = MOCKS[name]["mu_fc"]
    vmr_npz[f"{name}/mu_ml_truth"] = MOCKS[name]["mu_ml"]
    vmr_npz[f"{name}/mu_ml_map"] = spec["mu_ml_at_map"]
    vmr_npz[f"{name}/mu_fc_map"] = spec["mu_fc_at_map"]


summary = {
    "generated_utc": (
        datetime.datetime.now(datetime.timezone.utc)
        .isoformat(timespec="seconds").replace("+00:00", "Z")
    ),
    "bundle_path": str(BUNDLE_PATH.resolve()),
    "model": MODEL,
    "chemistry_type": str(bundle.chemistry_type),
    "model_type": str(bundle.model_type),
    "species_labels": list(species_labels),
    "params": list(SAMPLE_PARAMS),
    "nlayer": int(NLAYER),
    "noise_true": float(NOISE_TRUE),
    "pt_profile": {
        "family": "guillot_2010_piette_2019_alpha0",
        "log10_delta_pinned": float(LOG10_DELTA_PIN),
    },
    "nuts": {
        "num_warmup": int(NUM_WARMUP),
        "num_samples": int(NUM_SAMPLES),
        "max_tree_depth": int(MAX_TREE),
        "target_accept_prob": 0.8,
    },
    "priors": {
        "t_int": [200.0, 800.0], "t_eq": [1000.0, 2000.0],
        "log_gamma": [-1.5, 0.5], "logg": [4.0, 5.0],
        "RV": [35.0, 45.0], "vsini": [5.0, 15.0], "logZ": [-1.0, 1.0],
        "sigmain": "Exponential(1e-3)",
    },
    "anchors": {
        "solar_source": "src.constants.SOLAR_ABUNDANCES (Asplund 2009)",
        "logg_truth_cgs_log10": LOGG_TRUTH,
    },
    "per_truth": per_truth,
}

(OUTDIR / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
np.savez_compressed(OUTDIR / "posteriors.npz", **posteriors_npz)
np.savez_compressed(OUTDIR / "vmr_and_spectra.npz", **vmr_npz)

print()
print("wrote:")
for p in sorted(OUTDIR.iterdir()):
    print(f"  {p.name:<22s}  {p.stat().st_size / 1e3:7.1f} KB")


In [ ]:
# Replacement-claim headline.
print()
print("=" * 84)
print("REPLACEMENT-CLAIM HEADLINE  (truth = live FastChem; chain = emulator)")
print("=" * 84)
print(
    f"{'truth':<12}  {'logZ_bias':>10s}{'logZ_z':>9s}{'logZ_std':>10s}"
    f"  {'spec_chem':>10s}{'div':>5s}{'rms_map/n':>11s}"
)
print("-" * 84)
for T in TRUTHS:
    name = T["name"]
    pt = per_truth[name]
    s = pt["posterior_stats"]["logZ"]
    div = pt["nuts_extras_summary"].get("n_diverging", 0)
    rms_chem_n = pt["spectrum_rms_at_truth"]["ml_vs_fastchem_over_noise"]
    rms_map_n = pt["spectrum_rms_at_map"]["ml_vs_obs_over_noise"]
    print(
        f"{name:<12}  {s['bias_vs_truth']:>+10.3f}{s['z_vs_truth']:>+9.2f}"
        f"{s['std']:>10.3f}  {rms_chem_n:>10.3f}{div:>5d}{rms_map_n:>11.3f}"
    )
print("-" * 84)
print("Pass criteria for the replacement claim (the actual one):")
print("  |logZ_bias|  <= 0.10 dex     (unbiased recovery is the test)")
print("  |logZ_z|     <= 0.5 sigma    (within 1 posterior std of truth)")
print("  div          == 0            (no chain pathology)")
print()
print("Diagnostic context (does NOT by itself invalidate the claim):")
print("  spec_chem    rms(ml-fc)/noise at truth  -- chemistry residual scale")
print("  rms_map/n    rms(ml-obs)/noise at MAP   -- residual the chain absorbs")
print("=" * 84)
